# Model splitting for parallel and serial MODFLOW 6

**Model splitting** (also called *partitioning*) takes one MODFLOW 6 simulation and divides its grid into several smaller submodels that, run together, reproduce the original model's results. FloPy does this for you with the `Mf6Splitter()` class: it maps the original model's cell-to-cell connectivity, assigns every cell to a submodel using an array of model numbers you provide, and then builds the new submodels — automatically adding the **exchanges** (and any **movers**) that pass flow across the cuts so the pieces stay hydraulically connected.

Why split a model? Two main reasons:

- **Parallelization** — each submodel can be solved by a separate process (for example with Message Passing Interface, **MPI**), so a large simulation runs faster across multiple cores.
- **Memory** — no single process has to hold the entire grid in memory at once.

`Mf6Splitter()` works on groundwater flow (**GWF**) models, on combined flow-and-transport (**GWF/GWT**) models, and on Structured, Vertex, and Unstructured grids.

## What this notebook covers

You will:

- load and run an existing simulation, split it in two along a diagonal cut, and confirm the split results match the original;
- let `Mf6Splitter()` build a *load-balanced* partition automatically for a chosen number of submodels; and
- save and reload the **node mapping** so a split model's output can be stitched back onto the original grid for plotting and comparison.

The first example uses the classic Freyberg (1988) groundwater flow model.

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

## Example 1: splitting a simple structured grid model

This example shows the basics of using the `Mf6Splitter()` class and applies the method to the Freyberg (1988) model.

In [ ]:
simulation_ws = Path("../data/mf6-freyberg")

Load the existing Freyberg simulation from disk with `flopy.mf6.MFSimulation.load()`, pointing `sim_ws` at the folder that holds its input files. This gives you a complete simulation object to run and then split.

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=simulation_ws)

Point the simulation at a fresh working directory with `sim.set_sim_path()`, write the input files with `sim.write_simulation()`, and run MODFLOW 6 with `sim.run_simulation()`. Running the original model first gives you a baseline to compare the split results against.

In [ ]:
# point the loaded simulation at a fresh workspace, then write and run it
workspace = Path("models/freyberg")
sim.set_sim_path(workspace)
sim.write_simulation()
success, buff = sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

#### Plot the original model

Get the flow model from the simulation and load its final head array, then map it with `flopy.plot.PlotMapView()`. Draw the heads with `.plot_array()` and the boundary-condition cells — wells (`WELL`), river (`RIV`), and constant head (`CHD`) — with `.plot_bc()`. The `1e30` values MODFLOW writes for dry or inactive cells are replaced with `NaN` so they do not distort the color scale.

In [ ]:
# get the flow model; the plotting cell below loads its final heads
gwf = sim.get_model()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
pmv = flopy.plot.PlotMapView(gwf, ax=ax)
heads = gwf.output.head().get_alldata()[-1]
heads = np.where(heads == 1e30, np.nan, heads)
vmin = np.nanmin(heads)
vmax = np.nanmax(heads)
pc = pmv.plot_array(heads, vmin=vmin, vmax=vmax)
pmv.plot_bc("WEL")
pmv.plot_bc("RIV", color="c")
pmv.plot_bc("CHD")
pmv.plot_grid()
pmv.plot_ibound()
plt.colorbar(pc);

**What to look for.** This is the un-split, original model — your baseline. Heads grade across the domain, with the river and constant-head boundaries setting the levels and the well cells marking local stresses. Keep this map in mind: after splitting and recombining, the reassembled results should reproduce it.

### Creating an array that defines the new models

In order to split models, the model domain must be discretized using unique model numbers. Any number of models can be created, however all of the cells within each model must be contiguous.

The `Mf6Splitter()` class accept arrays that are equal in size to the number of cells per layer (`StructuredGrid` and `VertexGrid`) or the number of model nodes (`UnstructuredGrid`).

In this example, the model is split diagonally into two model domains.

In [ ]:
modelgrid = gwf.modelgrid

In [ ]:
array = np.ones((modelgrid.nrow, modelgrid.ncol), dtype=int)
ncol = 1
for row in range(modelgrid.nrow):
    if row != 0 and row % 2 == 0:
        ncol += 1
    array[row, ncol:] = 2

Preview how the domain will be divided by mapping the model-number array with `flopy.plot.PlotMapView().plot_array()`.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))
pmv = flopy.plot.PlotMapView(gwf, ax=ax)
pc = pmv.plot_array(array)
lc = pmv.plot_grid()
plt.colorbar(pc)
plt.show()

**What to look for.** This is the **split map** (partition map). Each color is one submodel, and the boundary between the colors is where `Mf6Splitter()` will cut the grid and insert exchanges. Notice that every color forms one contiguous block — a requirement for splitting.

### Splitting the model using `Mf6Splitter()`

The `Mf6Splitter()` class accepts one required parameter and one optional parameter. These parameters are:
   - `sim`: A flopy.mf6.MFSimulation object
   - `modelname`: optional, the name of the model being split. If omitted Mf6Splitter grabs the first groundwater flow model listed in the simulation

In [ ]:
# create the splitter for the loaded simulation
from flopy.mf6.utils import Mf6Splitter

mfs = Mf6Splitter(sim)

The model splitting is then performed by calling the `split_model()` function. `split_model()` accepts an array that is either the same size as the number of cells per layer (`StructuredGrid` and `VertexGrid`) model or the number of nodes in the model (`UnstructuredGrid`).

_Note:This method also accepts an optional `sim_ws` parameter. This parameter is useful when splitting a model that has external files and you'd like to preserve the external structure. The `sim_ws` parameter is the path to where you'd like to save the split simulation._

This function returns a new `MFSimulation` object that contains the split models and exchanges between them

In [ ]:
# split on the model-number array; returns a new simulation with both submodels
new_sim = mfs.split_model(array)
new_sim.set_sim_path(Path("models/freyberg_split"))

In [ ]:
# now to write and run the simulation
new_sim.write_simulation()
success, buff = new_sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

#### Compare the split models with the original

Get each submodel from the new simulation with `new_sim.get_model()`, load its final heads, and map the two side by side. Reuse the `vmin`/`vmax` from the original run so both panels share the baseline color scale and can be compared directly with the map above.

In [ ]:
# visualizing both models side by side
ml0 = new_sim.get_model("freyberg_1")
ml1 = new_sim.get_model("freyberg_2")

In [ ]:
heads0 = ml0.output.head().get_alldata()[-1]
heads1 = ml1.output.head().get_alldata()[-1]

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 6))
pmv = flopy.plot.PlotMapView(ml0, ax=ax0)
pmv.plot_array(heads0, vmin=vmin, vmax=vmax)
pmv.plot_ibound()
pmv.plot_grid()
pmv.plot_bc("WEL")
pmv.plot_bc("RIV", color="c")
pmv.plot_bc("CHD")
ax0.set_title("Model 0")

pmv = flopy.plot.PlotMapView(ml1, ax=ax1)
pc = pmv.plot_array(heads1, vmin=vmin, vmax=vmax)
pmv.plot_ibound()
pmv.plot_bc("WEL")
pmv.plot_bc("RIV", color="c")
pmv.plot_grid()
ax1.set_title("Model 1")

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
cbar = fig.colorbar(pc, cax=cbar_ax, label="Hydraulic heads")

**What to look for.** The two panels are the two submodels, solved together as one simulation. Read across the cut between them: the head pattern is continuous from one panel into the next — no jump or seam at the boundary — because the exchanges pass flow between the submodels. Side by side they reproduce the original single-model map, which is the whole point: splitting changes *how* the problem is solved, not the answer.

## Example 2: Create a load balanced splitting mask for a model

In the previous examples, the watershed model splitting mask was defined by the user. `Mf6Splitter` also has a method called `optimize_splitting_mask` that creates a mask based on the number of models the user would like to generate.

The `optimize_splitting_mask()` method generates a vertex weighted adjacency graph, based on the number active and inactive nodes in all layers of the model. This adjacency graph is then provided to `pymetis` which does the work for us and returns a membership array for each node.

The `optimize_splitting_mask()` method just needs the number of models supplied to it to split models however it does have a few optional parameters.

   - `nparts`: **required**, int; number of parts to split the model into (e.g., nparts=4 splits into 4 model domains)
   - `active_only`: optional, bool; calculate the split model based only on the active cells within the model domain
   - `options`: optional, pymetis.Options; method to pass through options to pymetis (e.g., options=pymetis.Options(seed=42))
   - `verbose`: optional, bool; prints progress statements if True

In [ ]:
# build a load-balanced partition automatically (uses pymetis under the hood)
mfs = Mf6Splitter(sim)
array = mfs.optimize_splitting_mask(nparts=5)

Map the optimized model-number array with `flopy.plot.PlotMapView().plot_array()` to preview the automatically generated partition.

In [ ]:
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid)
pc = pmv.plot_array(array, cmap="Dark2")
plt.colorbar(pc)

**What to look for.** `optimize_splitting_mask()` chose these partitions for you rather than you drawing them by hand. Each color is one submodel, and the pieces are sized to hold roughly equal numbers of active cells so the parallel workload stays balanced across processes.

Split the model on the optimized mask with `mfs.split_model()`, point the returned simulation at a workspace with `set_sim_path()`, then write and run it.

In [ ]:
sim_ws = Path("models/load_balanced_split")
new_sim = mfs.split_model(array)
new_sim.set_sim_path(sim_ws)
new_sim.write_simulation()
success, buff = new_sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

## Saving the node mapping to file

The **node mapping** records which original-grid cell each submodel cell came from. You need it to reassemble split output back onto the original grid, so it is worth saving alongside the split model. Save it with `Mf6Splitter.save_node_mapping()`, which writes the mapping to a Hierarchical Data Format version 5 (**HDF5**) binary file.

In [ ]:
# save the node mapping alongside the split model (HDF5)
mfs.save_node_mapping(sim_ws / "node_map.hdf5")

## Loading a saved node mapping from file

Reload a saved mapping — for example in a fresh session, after the split model has already been run — with `Mf6Splitter.load_node_mapping()`, passing it the HDF5 file written above. With the mapping loaded, the splitter can reconstruct arrays on the original grid without re-running the split.

In [ ]:
new_sim2 = flopy.mf6.MFSimulation.load(sim_ws=sim_ws)

# load_node_mapping is a staticmethod that returns a ready-to-use Mf6Splitter
mfsplit = Mf6Splitter.load_node_mapping(sim_ws / "node_map.hdf5")

#### Reassemble and plot the split results

Loop over the submodels, collecting each one's final head array into a dictionary keyed by submodel index, then stitch them back onto the original grid with `mfsplit.reconstruct_array()`. This is where the node mapping earns its keep: it tells `reconstruct_array()` which original cell each submodel cell belongs to. Plot the reconstructed head array on the original model grid.

In [ ]:
head_dict = {}
for ix, mname in enumerate(new_sim2.model_names):
    ml = new_sim2.get_model(mname)
    head_dict[ix] = ml.output.head().get_alldata()[-1]

ra_heads = mfsplit.reconstruct_array(head_dict)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 8))

pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pc = pmv.plot_array(ra_heads)
ib = pmv.plot_inactive()
plt.colorbar(pc, shrink=0.8);

**What to look for.** The submodel heads have been stitched back onto the single original grid, so this map should look like the baseline map from the start of the notebook — with no visible trace of where the submodel boundaries were. That agreement confirms the split-and-recombine round trip preserved the solution.

More information about the model splitter can be found [here](https://flopy.readthedocs.io/en/latest/Notebooks/mf6_parallel_model_splitting_example.html)

## Recap

In this notebook you:

- loaded and ran the original Freyberg (1988) flow model as a baseline;
- used `Mf6Splitter().split_model()` to split it into two submodels along a diagonal cut, letting FloPy add the exchanges that keep the pieces connected, and confirmed the split results match the original;
- had `optimize_splitting_mask()` build a *load-balanced* partition automatically for a chosen number of submodels; and
- saved the **node mapping** and used `reconstruct_array()` to stitch a split model's heads back onto the original grid for comparison.

The key idea: splitting changes *how* MODFLOW 6 solves a simulation — enabling parallel runs and lower per-process memory — without changing the answer.